# Gesture Drive AI


## Install dependencies


In [ ]:
%pip install pyserial mediapipe opencv-python


## Import libraries


In [ ]:
import time

import cv2
import mediapipe as mp
import serial


## Configuration


In [ ]:
ARDUINO_PORT = "COM12"
BAUD_RATE = 9600
CAMERA_INDEX = 0

previous_state = ""


## Gesture helpers


In [ ]:
def count_open_fingers(landmarks):
    fingers_open = 0

    if landmarks[8].y < landmarks[6].y:
        fingers_open += 1
    if landmarks[12].y < landmarks[10].y:
        fingers_open += 1
    if landmarks[16].y < landmarks[14].y:
        fingers_open += 1
    if landmarks[20].y < landmarks[18].y:
        fingers_open += 1

    return fingers_open


def classify_hand(landmarks):
    if count_open_fingers(landmarks) >= 3:
        return "OPEN"
    return "CLOSED"


## Connect to Arduino and camera


In [ ]:
arduino = serial.Serial(ARDUINO_PORT, BAUD_RATE)
time.sleep(2)

mp_hands = mp.solutions.hands
hands = mp_hands.Hands()
draw = mp.solutions.drawing_utils
cap = cv2.VideoCapture(CAMERA_INDEX)


## Run gesture detection


In [ ]:
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            continue

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)
        text = "NO HAND"

        if results.multi_hand_landmarks:
            for hand in results.multi_hand_landmarks:
                draw.draw_landmarks(frame, hand, mp_hands.HAND_CONNECTIONS)
                text = classify_hand(hand.landmark)

        if text != "NO HAND" and text != previous_state:
            arduino.write((text + "\n").encode())
            previous_state = text
            print("Sent:", text)

        cv2.putText(frame, text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow("Hand Detection", frame)

        if cv2.waitKey(1) == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    arduino.close()


## Manual cleanup


In [ ]:
if "cap" in globals():
    cap.release()

cv2.destroyAllWindows()

if "arduino" in globals() and arduino.is_open:
    arduino.close()
